# 06 Subhalo comparison

Collects the test scores of the subhalo runs into one table (MCC by run and spacing), and checks that the FiLM models actually use the spacing they are given: each is scored on the spacing-10 field while being told spacings from 3 to 10. If the scores barely change, the conditioning does nothing.

Section 0 re-scores older runs with a wider margin between prediction tiles. No training here. Runtime: **L4 GPU** with **High-RAM** (the sweep predicts the full box several times).

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/deepvoid
!git clone -q "https://{token}@github.com/skumagai3/deepvoid.git" /content/deepvoid
!git -C /content/deepvoid remote set-url origin https://github.com/skumagai3/deepvoid.git
!pip -q install --force-reinstall --no-deps /content/deepvoid
del token
!git -C /content/deepvoid log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Re-score finished runs

Scores every finished subhalo run the same way: a 32-cell margin between prediction tiles, the void-vs-rest scores, and a void threshold tuned on the validation slab. Results go to `test_scores_margin32.json` and `test_figure_margin32.png` in each run folder (the original scores are kept). About 3–5 minutes per run; runs already done are skipped.

In [ ]:
import json
from pathlib import Path
from deepvoid.train import evaluate_run

root = Path('/content/drive/MyDrive/deepvoid_runs')
RUNS = ['subhalo_A_scratch_L10', 'subhalo_B_finetune_L10', 'subhalo_C_mixed',
        'subhalo_A_smooth2ch_L10', 'subhalo_B_smooth2ch_L10', 'subhalo_C_mixed_smooth2ch',
        'subhalo_A_smoothonly_L10']
for run_name in RUNS:
    out = root / run_name
    path = out / 'test_scores_margin32.json'
    done = path.exists() and 'tuned' in next(iter(json.loads(path.read_text()).values()))
    if done:
        print('already done:', run_name)
    elif (out / 'best.keras').exists():
        print('re-scoring', run_name)
        evaluate_run(out, margin=32)


## 1. Test scores by run and spacing

Four-class MCC, then the void-vs-rest scores (the main measure for void science), with and without the tuned void threshold.

In [ ]:
import pandas as pd

def load_scores(run_name):
    for name in ('test_scores_margin32.json', 'test_scores.json'):
        path = root / run_name / name
        if path.exists():
            return json.loads(path.read_text())
    return None

tables = {'mcc (4 classes)': {}, 'void mcc': {}, 'void mcc, tuned threshold': {},
          'tuned threshold': {}, 'void precision, tuned': {}, 'void recall, tuned': {}}
for run_name in RUNS:
    scores = load_scores(run_name)
    if scores is None:
        print('not finished:', run_name)
        continue
    for name, s in scores.items():
        L = name.rsplit('/', 1)[-1]
        t = s.get('tuned', {})
        for key, value in (('mcc (4 classes)', s['mcc']), ('void mcc', s.get('void_mcc')),
                           ('void mcc, tuned threshold', t.get('void_mcc')),
                           ('tuned threshold', t.get('void_threshold')),
                           ('void precision, tuned', t.get('void_precision')),
                           ('void recall, tuned', t.get('void_recall'))):
            tables[key].setdefault(run_name, {})[L] = value

for key, rows in tables.items():
    print(f'\n{key}')
    print(pd.DataFrame(rows).T.reindex(columns=['L3', 'L5', 'L7', 'L10']).round(3).to_string())

from deepvoid.report import Report
rep = Report(root / 'subhalo_comparison')
rep.record(**{k.replace(' ', '_').replace(',', ''): v for k, v in tables.items()})


## 2. Spacing sweep for the FiLM runs

Takes a few minutes per run. The true spacing is 10.

In [ ]:
from deepvoid.train import spacing_sweep

told = [3, 5, 7, 10]
sweeps = {}
for run_name in ['subhalo_C_film', 'subhalo_C_film_curriculum']:
    if (root / run_name / 'best.keras').exists():
        result = spacing_sweep(root / run_name, 'tng300-3/subhalos/L10', told)
        sweeps[run_name] = {s: r['mcc'] for s, r in result.items()}
print(pd.DataFrame(sweeps).round(3))
rep.record(spacing_sweep_mcc=sweeps)

Tell Claude when it has finished; both tables are saved in `deepvoid_runs/subhalo_comparison/results.json`.